# Robust / Uncertain-input Classification

39番Notebookの標準binary GP contractを前提に、label noiseとcandidate-time input uncertaintyを比較します。


## 学習の目的

39番の標準GP分類を学んだ後に、**データの何が不確かなのか**を区別するNotebookです。ラベル（検査結果）を間違える問題と、指定した実験条件から実際の入力が揺れる問題は、異なるメカニズムです。

この例では0/1の合成ラベルを使用します。ラベルノイズモデルを構築していますが、デモの元データ自体に誤ラベルを人工注入しているわけではありません。

### このNotebookで学ぶこと

- **ラベルノイズ**と**候補入力の不確実性**を区別する
- `LabelNoiseBinarySingleTaskGPClassifier` と `ContinuousUncertainInputBinarySingleTaskGPClassifier` を使う
- 最新の公開 `fit()` でBernoulli ELBOを学習する
- 入力が揺らぐ場合のクラス確率をモンテカルロ積分する
- 入力不確実性を考慮した予測確率からEntropyを計算し、次のラベル取得候補を提案する
- 取得したラベルを追加して再学習する

**注意：** この例の入力不確実性は候補評価時の入力の揺らぎです。学習データの入力誤差を自動的に補正するモデルではありません。

In [ ]:
import torch

from robotorchan.models.classification import (
    ContinuousUncertainInputBinarySingleTaskGPClassifier,
    LabelNoiseBinarySingleTaskGPClassifier,
)

torch.set_default_dtype(torch.double)
torch.manual_seed(0)


## 1. 小さな合成データ

### 形状を確認する

`train_X` は8点×1変数、`train_Y` は8個の0/1ラベル、`grid` は予測位置51点です。合成ラベルは `x > 0.5` のとき1になります。実務の測定ではこの正解関数は未知です。

In [ ]:
train_X = torch.tensor([[0.05], [0.15], [0.30], [0.48], [0.52], [0.70], [0.85], [0.95]])
train_Y = (train_X.squeeze(-1) > 0.5).to(dtype=torch.double)
grid = torch.linspace(0.0, 1.0, 51).unsqueeze(-1)


## 2. Label noise と input uncertainty

Label noiseは観測過程、input uncertaintyはcandidate入力の不確かさです。

### モデルの違い

- `LabelNoiseBinarySingleTaskGPClassifier` はラベル観測過程に関する頑健性を扱います。
- `ContinuousUncertainInputBinarySingleTaskGPClassifier` は入力の揺らぎを考慮した推論用APIを備えます。

両者は同じデータで初期化されますが、*不確実性の原因* を同一視していません。

### ラベルノイズモデルが表す確率

`LabelNoiseBinarySingleTaskGPClassifier` は観測ラベルの反転確率を明示的に扱います。たとえば偽陽性率は「本来クラス0なのに観測ラベルが1になる確率」、偽陰性率はその逆です。

`predict_clean_proba(X)` はラベル反転前のクラス確率、`predict_proba(X)` はラベル反転を含む観測ラベルの確率を表します。**両者を混同しないでください。** デフォルトの反転率は0なので、次のセルでは説明用に反転率を指定した例も作ります。

In [ ]:
label_noise_model = LabelNoiseBinarySingleTaskGPClassifier(train_X, train_Y)
uncertain_model = ContinuousUncertainInputBinarySingleTaskGPClassifier(train_X, train_Y)
print(label_noise_model.classification_metadata)
print(uncertain_model.classification_metadata)


In [ ]:
# 既存モデルはそのままにし、観測ラベル反転率を指定した例を別に作る
flip_example = LabelNoiseBinarySingleTaskGPClassifier(
    train_X, train_Y, false_positive_rate=0.10, false_negative_rate=0.05
)
print("false-positive / false-negative:", flip_example.flip_probabilities.detach())
with torch.no_grad():
    clean_probability = flip_example.predict_clean_proba(grid)[..., 1]
    observed_probability = flip_example.predict_proba(grid)[..., 1]
print("clean vs observed probability at x=0.5:",
      clean_probability[len(grid)//2].item(),
      observed_probability[len(grid)//2].item())

## 3. Candidate-time input uncertainty

### input_std と num_samples

`input_std=0.03` は連続入力の標準偏差を仮定します。入力 `x=0.5` であれば、実際の入力がその付近で揺らぐことを想定します。

`num_samples=32` は揺らいだ入力を扱う近似計算で用いるサンプル数です。有限サンプルのため予測は変動します。

**重要：** ここでの計算は学習前のモデルに対して行います。意味のある学習後の比較は次のセクションで行います。

In [ ]:
input_std = torch.full_like(grid, 0.03)
with torch.no_grad():
    probability = uncertain_model.predict_proba_with_input_uncertainty(
        grid, input_std=input_std, num_samples=32
    )
print(probability.shape)


### 入力の揺らぎをどう積分するか

候補を `x` に設定しても、実際の入力が `x + ε` に変動する状況を考えます。`ε` は平均0、標準偏差 `input_std` のガウスノイズです。

`predict_proba_with_input_uncertainty()` は揺らいだ入力をサンプリングし、**各入力での予測クラス確率を平均**します。これは潜在関数の標準偏差をそのままクラス確率へ足す操作ではありません。

`num_samples` が有限なのでモンテカルロ誤差があり、実行ごとに結果が少し変わる場合があります。

## 4. 使い分け

- 誤ラベルはrobust likelihood側で扱います。
- candidate Xの揺らぎはuncertain-input側で扱います。
- Student-t regression likelihoodをbinary labelへ機械的に移植しません。
- categorical input uncertaintyには専用のexact marginalization APIがあります。

## 5. 学習・候補提案・ラベル評価

両モデルを既存の変分ELBOで学習し、入力不確実性を考慮した予測確率に基づいて、未ラベル候補を選択します。候補の選択は有限プールで行います。


### ELBO学習と候補選択の考え方

2つのモデルを同様の学習ループで更新し、入力の揺らぎを考慮した**クラス1予測確率**から二値エントロピーを求めています。エントロピーが大きい候補（ラベルが曖昧な候補）を未ラベル候補の有限集合から選びます。

ここでは **有限プールでの単純な最大値選択** であり、39番の `optimize_acqf` と同じ方法ではありません。

`observed_label` は合成関数から計算した疑似的な測定結果です。実際の設備では実験後にラベルが得られます。

### 最新の `fit()` を使う理由

これらのモデルは二値変分GP分類器の派生モデルです。現在の公開API `model.fit(epochs=..., lr=...)` で学習できるため、Notebook側でELBOやAdamループを再実装する必要はありません。

**候補提案の考え方：** `predict_proba_with_input_uncertainty()` で入力揺らぎを積分した予測確率から、Bernoulli Entropy `-p log p -(1-p) log(1-p)` を計算します。これは入力不確実性を考慮した**カスタムの有限候補スコア**です。標準の `PredictiveEntropy(model)` をそのまま使うと、入力揺らぎの積分は含まれません。

In [ ]:
# 両モデルとも公開 fit() API を利用してBernoulli ELBOを学習する
for model in (label_noise_model, uncertain_model):
    model.fit(epochs=20, lr=0.05)
    model.eval()

candidate_pool = torch.linspace(0.0, 1.0, 101).unsqueeze(-1)
with torch.no_grad():
    probabilities = uncertain_model.predict_proba_with_input_uncertainty(
        candidate_pool,
        input_std=torch.full_like(candidate_pool, 0.03),
        num_samples=32,
    )
    positive_probability = probabilities[..., 1]
    entropy = -(
        positive_probability.clamp(1e-8, 1 - 1e-8).log() * positive_probability
        + (1 - positive_probability).clamp(1e-8, 1 - 1e-8).log()
        * (1 - positive_probability)
    )
candidate = candidate_pool[entropy.argmax()].unsqueeze(0)
observed_label = (candidate.squeeze(-1) > 0.5).long()
print("candidate:", candidate, "observed label:", observed_label)


### 予測値の意味と探索の限界

`entropy.argmax()` は101点の候補プールからEntropy最大の1点を選びます。これはクラス1確率の最大化ではなく、**ラベルが曖昧な点を優先するActive Learning**です。

候補プールの外に最良点がある可能性は残ります。また、モンテカルロサンプル数を変えると選択点が変化することがあります。入力揺らぎを積分する専用のBoTorch獲得関数を構築したわけではないため、既存の有限候補評価を維持します。

## 6. 図で比較する：通常の確率と入力の揺らぎを考慮した確率

学習済み `uncertain_model` のクラス1確率について、入力の揺らぎを考慮しない場合と考慮した場合を並べて表示します。

このグラフは**入力不確実性の効果を見るための説明図**です。ラベルノイズモデルとどちらが高性能かを検証する実験ではありません。

In [ ]:
import matplotlib.pyplot as plt

with torch.no_grad():
    base_probability = uncertain_model.predict_proba(candidate_pool)[..., 1]
    uncertain_probability = uncertain_model.predict_proba_with_input_uncertainty(
        candidate_pool,
        input_std=torch.full_like(candidate_pool, 0.03),
        num_samples=64,
    )[..., 1]

xs = candidate_pool.squeeze(-1).detach().cpu().numpy()
base_values = base_probability.detach().cpu().numpy()
uncertain_values = uncertain_probability.detach().cpu().numpy()
entropy_values = entropy.detach().cpu().numpy()
selected_x = candidate.detach().cpu().reshape(-1)[0].item()

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].plot(xs, base_values, label="Without input uncertainty")
axes[0].plot(xs, uncertain_values, label="With input uncertainty")
axes[0].scatter(
    train_X.squeeze(-1).detach().cpu().numpy(),
    train_Y.detach().cpu().numpy(),
    label="Observed labels",
)
axes[0].set_ylim(-0.05, 1.05)
axes[0].set_ylabel("P(y=1)")
axes[0].legend()
axes[1].plot(xs, entropy_values, label="Entropy (32 samples)")
axes[1].axvline(selected_x, linestyle="--", color="gray", label="Selected")
axes[1].set_ylabel("Entropy")
axes[1].set_xlabel("Input x")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### 図からの読み取り

1. 入力揺らぎを加えると、確率曲線がどこで変化するかを確認します。変化の向きや程度は学習結果に依存します。
2. エントロピー最大の場所は、必ずしも入力不確実性を考慮する前と同じとは限りません。
3. サンプル数を変えて再実行し、Monte Carlo誤差の影響を確認してみてください。

## 7. 観測ラベルを追加して再学習する

候補提案だけではActive Learningは完結しません。合成関数で得た `observed_label` を学習データに追加し、入力不確実性モデルを公開 `fit()` で再学習します。

実際の運用では、ここでラベルを取得するために検査・測定・人手評価を行います。

In [ ]:
updated_X = torch.cat((train_X, candidate.detach()), dim=0)
updated_Y = torch.cat((train_Y, observed_label.to(train_Y.dtype).reshape(-1)), dim=0)
assert updated_X.shape == (len(train_X) + 1, 1)
assert updated_Y.shape == (len(train_Y) + 1,)

updated_model = ContinuousUncertainInputBinarySingleTaskGPClassifier(updated_X, updated_Y)
updated_model.fit(epochs=20, lr=0.05)
updated_model.eval()
with torch.no_grad():
    updated_probability = updated_model.predict_proba_with_input_uncertainty(
        candidate_pool,
        input_std=torch.full_like(candidate_pool, 0.03),
        num_samples=64,
    )[..., 1]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(xs, uncertain_values, label="before new label")
ax.plot(xs, updated_probability.detach().cpu().numpy(), label="after new label")
ax.axvline(selected_x, linestyle="--", color="gray", label="newly labeled x")
ax.set(xlabel="input x", ylabel="P(y=1) with input uncertainty",
       title="Active Learning: before / after one label")
ax.legend()
plt.tight_layout()
plt.show()

## 初学者向け補足：何が不確かなのかを区別する

同じ「不確実性」でも、原因が異なると対策も異なります。

| 状況 | 例 | 主な対処 |
|---|---|---|
| ラベルノイズ | 検査員ごとに合否判定が揺れる | ラベル生成・観測過程の頑健化 |
| 入力不確実性 | 設定温度と実温度にずれがある | 入力の揺らぎを考慮した予測 |
| モデル不確実性 | 未測定の領域が広い | 追加観測・Active Learning |

### Notebookの読み進め方

1. 合成データの0/1ラベルを確認します。
2. どのモデルがラベルノイズを扱い、どの処理が入力の揺らぎを扱うかを追います。
3. 学習後の予測確率を比較し、入力が揺れたときに候補の評価が変わるかを確認します。
4. 候補提案後に得られるラベルは実験・測定で初めて判明することを意識します。合成データの真値を使うのはデモ用です。

**注意：** ラベルノイズへの頑健性と、未知の入力変動への頑健性は同義ではありません。例の少ないepochや小さな合成データから、実問題での優劣を結論づけないでください。

### 学習・探索・更新の対応

| 処理 | 公開API・方法 |
|---|---|
| ラベルノイズ | `LabelNoiseBinarySingleTaskGPClassifier` |
| 入力不確実性 | `ContinuousUncertainInputBinarySingleTaskGPClassifier` |
| 学習 | `model.fit(epochs=..., lr=...)` |
| 観測前の確率 | `model.predict_proba()` |
| ラベル反転前の確率 | `label_noise_model.predict_clean_proba()` |
| 入力揺らぎを積分した確率 | `predict_proba_with_input_uncertainty()` |
| 候補提案 | 入力揺らぎを積分した確率のEntropy最大化（有限候補） |
| 観測更新 | `torch.cat()` と新モデルの `fit()` |

**分類の不確実性には複数の原因があります。** モデルが何を積分し、どの確率を返すかを理解して獲得関数を選んでください。